# M1 Pro Basic Control

All positions in this notebook use **robot frame** coordinates `(x, y, z, r)` in mm and degrees,
exactly as the Dobot controller expects them.

M1 Pro workspace limits:
- Z: 5 – 245 mm
- Forward (x ≥ 0): radial reach 153 – 400 mm from base
- R: end-effector rotation in degrees

Verify coordinates look sensible before setting `simulation = False`.

In [ ]:
from dobot_driver import M1Pro

DOBOT_IP   = "192.168.2.6"
SIMULATION = False          # set True to test without moving the real arm

# ── Robot-frame saved positions (x mm, y mm, z mm, r deg) ─────────────────
# Adjust these values to match your actual setup.
# Use arm.get_pose() to read back the current raw robot coordinates.
SAVED = {
    "home":               [320,   0, 240, 20],
    "zone1_tray_1":       [300,   0, 100, 20],   # directly below home
    "zone1_tray_2":       [250, 100,  80, 20],   # r≈269, front-right
    "zone1_tray_4":       [200, 200,  80, 20],   # r≈283, further right
    "zone2_tray_center":  [350, -80, 100, 20],   # r≈359, front-left
}

SAFE_HEIGHT = 240   # mm – arm rises to this Z before lateral moves

In [ ]:
arm = M1Pro(
    DOBOT_IP,
    simulation=SIMULATION,
    safe_height=SAFE_HEIGHT,
    home_position=[250, 0, 240, 20],   # raw robot home pose
    calibrated_offset=[[0, 0, 0], [0, 0, 0]],  # no work transform needed
    tool_offset=[[0, 0, 0], [0, 0, 0]],         # no tool offset needed
)

print("Connected. Current robot pose:")

In [ ]:
# ── Get current position ──────────────────────────────────────────────────
pose = arm.get_pose()
joints = arm.get_joint_angles()

print(f"Robot pose  →  x={pose.x:.3f} mm  y={pose.y:.3f} mm  z={pose.z:.3f} mm  r={pose.r:.3f}°")
print(f"Joint angles → {', '.join(f'J{i+1}={v:.3f}°' for i, v in enumerate(joints))}")

In [ ]:
arm.reset()

In [ ]:
# ── Home ──────────────────────────────────────────────────────────────────
arm.home()
print("After home:", arm.get_pose())

In [ ]:
# ── Preview targets before moving ─────────────────────────────────────────
for name in SAVED:
    info = arm.preview_target(name)
    status = "OK" if info["within_workspace"] else "OUT OF RANGE"
    print(f"{name:25s}  {status:13s}  {info['robot_pose']}")

In [ ]:
# ── Direct move to a named position (robot frame) ─────────────────────────

arm.move("zone1_tray_1")
print("After move:", arm.get_pose())

In [ ]:
# ── Direct move to explicit coordinates ───────────────────────────────────

arm.safe_move([263, -217, 80, -70])


In [ ]:

arm.pick_from(position=[283, -177, 60, -70])

In [ ]:
arm._move([283, -177, 100, 20])

In [ ]:
arm.place_to(position=[214, -77, 82, 20])

In [ ]:

arm.set_speed_factor(0.1)
arm._move([268, -77, 60, 20])

In [ ]:
arm.pick_from(position=[283, -177, 60, -70])

In [ ]:
arm._move(position=[228, -54, 100, 80])

In [ ]:
arm.place_to(position=[268, -77, 82, 20])

In [ ]:
arm.open_gripper()

In [ ]:
arm.close_gripper()

In [ ]:
arm.set_speed_factor(0.1)
arm.close_gripper()

arm._move([283, -217, 80, -70])

In [ ]:
# ── Gripper control ───────────────────────────────────────────────────────
# Move to pick position, close gripper, rise to safe height, move to place
# position, open gripper.

arm.safe_move("zone1_tray_1", speed_factor_lateral=0.3)

arm.close_gripper()
print("Gripper closed")

arm.safe_move("zone2_tray_center", speed_factor_lateral=0.3)

arm.open_gripper()
print("Gripper opened")

print("After pick-and-place:", arm.get_pose())

In [ ]:
# Centrifuge 1 North pick

arm.open_gripper()
arm.safe_move([232,235.7,42.5,-70])
arm.close_gripper()
arm._move([232,235.7,82.5,-70])

In [ ]:
#Centrifuge 1 North place
arm.safe_move([232,237.7,42.5,-70])
arm.open_gripper()

In [ ]:
# Centrifuge 1 South pick

arm.open_gripper()
arm.safe_move([232,189.7,42.5,-70])
arm.close_gripper()
arm._move([232,189.7,82.5,-70])

In [ ]:
#Centrifuge 1 South place

arm.safe_move([232,187.7,42.5,-70])
arm.open_gripper()

In [ ]:
#position A1
arm.open_gripper()
print("Gripper closed")
arm.safe_move([287, -234, 20, -33])
#open gripper



In [ ]:
#position A1
arm.safe_move([287, -234, 30, -33])
#open gripper
arm.close_gripper()
print("Gripper closed")


In [ ]:
#position A1
arm.move([287, -234, 10, -33])
#open gripper
arm.close_gripper()
print("Gripper closed")



In [ ]:
arm.open_gripper()


In [ ]:
arm.close_gripper()
print("Gripper closed")

In [ ]:
#position D1
arm._move([286, -176, 62, -33])

In [ ]:
#position A6
arm._move([182, -174, 5, -33])

In [ ]:
#position D6
arm.safe_move([182, -234, 5, -33])

arm.open_gripper()
print("Gripper opened")

In [ ]:
# ── Safe move (rise → lateral → descend) to a named position ──────────────
arm.safe_move("zone1_tray_2", speed_factor_lateral=0.2)
print("After safe_move:", arm.get_pose())

In [ ]:
# ── Safe move to explicit coordinates ─────────────────────────────────────
arm.safe_move([200, 200, 80, -33], speed_factor_lateral=0.2)
print("After explicit safe_move:", arm.get_pose())

In [ ]:
# ── Return home and clean up ───────────────────────────────────────────────
arm.home()
arm.reset()

In [ ]:
arm._disconnect()

In [ ]:
arm.home()
print("After home:", arm.get_pose())